# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати пам'яті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

In [ ]:
import time
from IPython import get_ipython
_cell_times, _t_start = [], time.time()
def _pre_cell(info):
    globals()["_t_cell"] = time.time()
def _post_cell(result):
    lines = [l for l in (result.info.raw_cell or "").splitlines() if l.strip() and not l.lstrip().startswith("#")]
    _cell_times.append((lines[0][:80] if lines else "", round(time.time() - globals().get("_t_cell", time.time()), 1)))
get_ipython().events.register("pre_run_cell", _pre_cell)
get_ipython().events.register("post_run_cell", _post_cell)

In [ ]:
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "bitsandbytes" "datasets" "accelerate"
!pip uninstall -q -y torchao

In [ ]:
import gc
import json
import os
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Для цього ноутбука потрібна GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Пам'ять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
Пам'ять GPU: 15.6 GB


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) -
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` - відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` - конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [ ]:
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}


In [ ]:
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [ ]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` - коротка інструкція;
- `user` - повідомлення клієнта;
- `assistant` - правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [ ]:
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

### Тренувальний, валідаційний і тестовий набори

- **Тренувальний** (2800 прикладів) - на ньому модель вчиться.
- **Валідаційний** (200) - на ньому під час тренування обчислюється validation loss, щоб помітити перенавчання.
- **Тестовий** (1000) - на ньому лише один раз, після тренування, рахуємо підсумкові метрики.
  Під час тренування й налаштування ці приклади не використовуються, тому оцінка на них найчесніша.

Після дедуплікації та перемішування тестовий набір беремо з прикладів, які не потрапили до перших 3000.

Тестовий набір більший за валідаційний, бо від його розміру залежить точність вимірювання метрик:
на 1000 прикладах одна помилка змінює точність на 0,1%, тоді як на 200 - на 0,5%.
Донавчені моделі помиляються рідко, і на малому тестовому наборі різниця між ними може бути непомітною.

In [ ]:
shuffled = dedup_dataset.shuffle(seed=42)

dataset = shuffled.select(range(3000)).map(to_messages, remove_columns=shuffled.column_names)
dataset = dataset.train_test_split(test_size=200, seed=42)
train_dataset, val_dataset = dataset["train"], dataset["test"]
test_dataset = shuffled.select(range(3000, 4000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
train_dataset[0]["messages"]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'content': 'Classify the customer message. Reply with JSON only.',
  'role': 'system'},
 {'content': 'I have got to locate hte bills from {{Person Name}}',
  'role': 'user'},
 {'content': '{"category": "INVOICE", "intent": "check_invoice"}',
  'role': 'assistant'}]

## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сім'ї з однаковою архітектурою й даними попереднього тренування - **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [ ]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки "до" передаємо його в промпті разом з описом формату.

In [ ]:
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів


In [ ]:
sample_msgs = [ex["messages"][1]["content"] for ex in test_dataset.select(range(200))]
def prompt_tokens(system_prompt):
    return sum(
        len(tokenizer(tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": m}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False))["input_ids"])
        for m in sample_msgs) / len(sample_msgs)

short_in, long_in = prompt_tokens(SHORT_PROMPT), prompt_tokens(LONG_PROMPT)
print(f"Вхід із коротким промптом: {short_in:.0f} токенів, із детальним: {long_in:.0f} токенів")
print(f"Різниця на 10 000 звернень на день: {(long_in - short_in) * 10_000 / 1e6:.2f} млн токенів/день, "
      f"{(long_in - short_in) * 10_000 * 30 / 1e6:.1f} млн токенів/місяць")

Вхід із коротким промптом: 38 токенів, із детальним: 193 токенів
Різниця на 10 000 звернень на день: 1.55 млн токенів/день, 46.5 млн токенів/місяць


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [ ]:
def classify_batch(model, messages_batch, system_prompt, max_new_tokens=60):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар 'звернення - відповідь моделі'."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє пам'ять GPU після видалення моделей."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` - ранг матриць адаптера: що більший, то більше параметрів і "свободи" має адаптер;
- `lora_alpha` - масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` - dropout в адаптері для регуляризації;
- `target_modules="all-linear"` - адаптери додаються до всіх лінійних шарів (attention і MLP).

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

Параметри тренування (`SFTConfig`):
- `assistant_only_loss=True` - loss рахується лише на відповідях асистента, а не на репліках користувача й системному промпті:
  для решти токенів мітки замінюються значенням −100, і функція втрат їх пропускає;
- `max_steps=100` - кількість кроків тренування; за один крок модель бачить 8 прикладів, тобто за все тренування - 800 із 2800,
  менше третини епохи. Для вузької задачі цього достатньо: validation loss виходить на плато раніше;
- `learning_rate=2e-4` - для LoRA learning rate приблизно в 10 разів більший, ніж для повного донавчання;
- `eval_strategy="steps"` і `eval_steps=10` - кожні 10 кроків модель оцінюється на валідаційному наборі:
  якщо validation loss почне зростати, модель перенавчається;
- `fp16=True` - змішана точність (T4 не підтримує bf16);
- `gradient_checkpointing=True` - активації не зберігаються, а перераховуються під час зворотного проходу: повільніше, але потрібно менше пам'яті.

In [ ]:
def make_config(output_dir, learning_rate=2e-4):
    return SFTConfig(
        output_dir=output_dir,
        max_length=256,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        max_steps=100,
        eval_strategy="steps",
        eval_steps=10,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )


def train(model, output_dir, peft_config=peft_config, learning_rate=2e-4):
    """Тренує модель (з LoRA, якщо задано peft_config) і повертає тренер, тривалість, пікову пам'ять GPU та validation loss."""
    torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(
        model=model,
        args=make_config(output_dir, learning_rate),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    if getattr(model, "is_loaded_in_4bit", False):
        for param in trainer.model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
    if peft_config is not None:
        trainer.model.print_trainable_parameters()
    start = time.time()
    trainer.train()
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    stats = {
        "Тривалість, хв": (time.time() - start) / 60,
        "Пікова пам'ять GPU, GB": torch.cuda.max_memory_allocated() / 1e9,
        "Validation loss": eval_losses[-1],
    }
    return trainer, stats

## 5. LoRA на моделях різного розміру

Для кожної моделі виконуємо однакові кроки:
1. завантажуємо модель у fp16 (ваги Qwen3 збережено в bfloat16, але T4 не підтримує його апаратно);
2. оцінюємо її **до донавчання** з детальним промптом, у якому перелічено всі мітки;
3. донавчаємо методом LoRA з однаковими параметрами;
4. оцінюємо **після донавчання** з коротким промптом без переліку міток;
5. зберігаємо адаптер і вивантажуємо модель з GPU, щоб звільнити пам'ять для наступної.

In [ ]:
results = {}


def run_lora(model_name):
    """Повний цикл для однієї моделі: оцінка до, LoRA, оцінка після, збереження адаптера."""
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    print(f"{short_name}: {model.num_parameters() / 1e9:.2f} млрд параметрів, "
          f"пам'ять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("\nДо донавчання:")
    show(before_answers, k=3)

    trainer, stats = train(model, f"{short_name}-lora")
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("\nПісля донавчання:")
    show(after_answers, k=3)

    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1e6

    results[short_name] = {"до": before_metrics, "LoRA": after_metrics | stats}
    del trainer, model
    free_memory()
    return pd.DataFrame(results[short_name]).round(2)

In [16]:
run_lora("Qwen/Qwen3-0.6B")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-0.6B: 0.60 млрд параметрів, пам'ять GPU після завантаження: 1.2 GB

До донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (8 outputs left out)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.362490,0.478024,0.437590,0.894284,4323.000000
20,0.321075,0.142579,0.187329,0.956564,8672.000000
30,0.080716,0.060280,0.072056,0.986241,13032.000000
40,0.059501,0.024369,0.043479,0.991889,17377.000000
50,0.022630,0.010941,0.027282,0.997168,21714.000000
60,0.006531,0.007179,0.018348,0.998415,26070.000000
70,0.011597,0.006614,0.016946,0.997814,30452.000000


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.362490,0.478024,0.437590,0.894284,4323.000000
20,0.321075,0.142579,0.187329,0.956564,8672.000000
30,0.080716,0.060280,0.072056,0.986241,13032.000000
40,0.059501,0.024369,0.043479,0.991889,17377.000000
50,0.022630,0.010941,0.027282,0.997168,21714.000000
60,0.006531,0.007179,0.018348,0.998415,26070.000000
70,0.011597,0.006614,0.016946,0.997814,30452.000000
80,0.008151,0.005353,0.013802,0.998745,34813.000000
90,0.005143,0.005575,0.013733,0.998755,39116.000000
100,0.015742,0.005505,0.013629,0.998755,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,97.90
"Тривалість, хв",NaN,2.59
"Пікова пам'ять GPU, GB",NaN,1.42
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


In [17]:
run_lora("Qwen/Qwen3-1.7B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.993839,0.438071,0.369675,0.905950,4323.000000
20,0.315746,0.133755,0.193267,0.955675,8672.000000
30,0.080187,0.045619,0.050167,0.988109,13032.000000
40,0.049633,0.024996,0.029017,0.990317,17377.000000
50,0.017502,0.017397,0.025820,0.994078,21714.000000
60,0.012237,0.005393,0.014650,0.998740,26070.000000
70,0.013269,0.005115,0.011206,0.997189,30452.000000
80,0.010280,0.004942,0.010459,0.997494,34813.000000
90,0.013139,0.004111,0.010383,0.997812,39116.000000
100,0.009742,0.003938,0.010325,0.997812,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",62.7,99.30
"Точність intent, %",57.4,96.60
"Тривалість, хв",NaN,2.47
"Пікова пам'ять GPU, GB",NaN,3.83
Validation loss,NaN,0.00
"Адаптер, MB",NaN,69.78


In [ ]:
run_lora("Qwen/Qwen3-4B")

### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, пам'ять і розмір адаптера.

In [19]:
pd.DataFrame({
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00
"Точність intent, %",25.6,97.90,57.4,96.60,72.5,97.70
"Тривалість, хв",NaN,2.59,NaN,2.47,NaN,3.67
"Пікова пам'ять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19


## 6. Повне донавчання Qwen3-0.6B

Модель на 0,6 млрд параметрів потребує для повного донавчання близько 10 GB, тому на T4 її можна донавчити повністю й порівняти з LoRA.

Відмінності від LoRA:
- `peft_config=None` - тренуються всі ваги моделі;
- модель завантажуємо у **fp32**: змішана точність fp16 тримає головну копію ваг у fp32, а в fp16 лише обчислює прохід;
- `learning_rate=2e-5` - для повного донавчання learning rate приблизно в 10 разів менший, ніж для LoRA.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map="auto")
trainer, full_stats = train(model, "Qwen3-0.6B-full", peft_config=None, learning_rate=2e-5)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)

del trainer, model
free_memory()

pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"], "Qwen3-0.6B повне": full_metrics | full_stats}).round(2)

## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [21]:
adapter_dir = "Qwen3-4B-lora-adapter"
for name in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, name))
    print(f"{name:<35} {size / 1e6:>8.1f} MB")
print("\nДля порівняння, ваги Qwen3-4B у fp16: ~8 GB")

README.md                                0.0 MB
adapter_config.json                      0.0 MB
adapter_model.safetensors              132.2 MB
chat_template.jinja                      0.0 MB
tokenizer.json                          11.4 MB
tokenizer_config.json                    0.0 MB
training_args.bin                        0.0 MB

Для порівняння, ваги Qwen3-4B у fp16: ~8 GB


Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [ ]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
print(type(model).__name__)


Перевіримо злиту модель на власних повідомленнях:

In [ ]:
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

In [24]:
del model
free_memory()

## 8. QLoRA

QLoRA завантажує заморожені ваги Qwen3-4B в **4 бітах** (формат NF4), а адаптери LoRA тренує у звичайній точності.
Код тренування той самий, змінюється лише завантаження моделі.

### Формат NF4

У 4 бітах вага може набувати лише одного з 16 значень. Ваги ділять на блоки по 64, і для кожного блоку
зберігають масштаб - найбільше за модулем значення в блоці. Формати відрізняються тим, як розташовані ці 16 значень:
- **INT4** - 16 рівновіддалених значень. Ваги нейромережі розподілені приблизно нормально: більшість близька до нуля,
  а великих за модулем мало. Тому крайні значення використовуються рідко, а біля нуля їх бракує;
- **NF4 (NormalFloat4)** - 16 значень розташовані за квантилями нормального розподілу: густо біля нуля й рідко на краях.
  Кожне значення використовується приблизно однаково часто, тому помилка квантизації менша, ніж у INT4.

NF4 - формат лише для зберігання: перед кожним множенням ваги деквантизуються у fp16 (`bnb_4bit_compute_dtype`).
Детальніше - у блозі Hugging Face [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)
і в статті [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314).

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", quantization_config=bnb_config, device_map="auto")
print(f"Пам'ять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
trainer, qlora_stats = train(model, "Qwen3-4B-qlora")
qlora_metrics, _ = evaluate(trainer.model, SHORT_PROMPT)
qlora_stats

## 9. Підсумкове порівняння

In [27]:
summary = {
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary["Qwen3-4B QLoRA"] = qlora_metrics | qlora_stats
pd.DataFrame(summary).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA,Qwen3-0.6B повне,Qwen3-4B QLoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00,99.50,99.90
"Точність intent, %",25.6,97.90,57.4,96.60,72.5,97.70,98.30,97.80
"Тривалість, хв",NaN,2.59,NaN,2.47,NaN,3.67,1.67,4.93
"Пікова пам'ять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75,10.80,3.67
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00,0.00,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19,NaN,NaN


**Що варто помітити:**
- LoRA тренує близько 1-2% параметрів моделі, а адаптер займає десятки або сотні мегабайт замість гігабайтів для всієї моделі.
- Порівняйте моделі різного розміру: наскільки менша модель після донавчання поступається більшій і скільки вона заощаджує часу й пам'яті.
  Для вузької задачі з тисячами прикладів донавчена мала модель може бути вигіднішою за велику.
- Порівняйте повне донавчання Qwen3-0.6B з LoRA на тій самій моделі за якістю, часом і пам'яттю.
- До донавчання моделі отримують детальний промпт із переліком міток, після - лише короткий:
  формат і значення міток тепер зберігаються у вагах, а не в промпті. Коротший промпт означає швидшу й дешевшу обробку кожного запиту.
- QLoRA потребує помітно менше пам'яті, ніж LoRA на тій самій моделі, але тренується повільніше, бо ваги щоразу деквантизуються під час обчислень.

### Тривалість виконання клітинок

In [28]:
import pandas as pd
print(f"Увесь ноутбук: {(time.time() - _t_start) / 60:.1f} хв")
pd.DataFrame(_cell_times, columns=["клітинка (перший рядок коду)", "секунд"]).sort_values("секунд", ascending=False).head(10)

Увесь ноутбук: 32.1 хв


,клітинка (перший рядок коду),секунд
17,"run_lora(""Qwen/Qwen3-4B"")",575.1
25,"trainer, qlora_stats = train(model, ""Qwen3-4B-...",434.1
16,"run_lora(""Qwen/Qwen3-1.7B"")",332.2
15,"run_lora(""Qwen/Qwen3-0.6B"")",297.5
19,"model = AutoModelForCausalLM.from_pretrained(""...",149.2
2,import gc,50.8
21,from peft import PeftModel,28.8
24,bnb_config = BitsAndBytesConfig(,26.3
1,"!pip install -q ""transformers>=4.55.0"" ""trl>=1...",15.1
8,"MODELS = [""Qwen/Qwen3-0.6B"", ""Qwen/Qwen3-1.7B""...",4.7


---
# Відповіді на питання 0.1 (22-30)

**22. Як довго виконувалися клітинки ноутбуку `03_sft_lora`? Чи були проблеми з запуском, які?**

Весь ноутбук (повний запуск після Restart → Run all) виконувався **32,1 хв** на T4. Найдовші клітинки:

| клітинка | секунд | хв |
|---|---:|---:|
| `run_lora("Qwen/Qwen3-4B")` - оцінка "до", LoRA, оцінка "після" | 575 | 9,6 |
| `train(model, "Qwen3-4B-qlora")` - тренування й оцінка QLoRA | 434 | 7,2 |
| `run_lora("Qwen/Qwen3-1.7B")` | 332 | 5,5 |
| `run_lora("Qwen/Qwen3-0.6B")` | 298 | 5,0 |
| повне донавчання Qwen3-0.6B | 149 | 2,5 |
| імпорти (`import gc …`) | 51 | 0,8 |
| завантаження адаптера з диска (`PeftModel`) | 29 | 0,5 |
| завантаження 4B у 4 бітах (`BitsAndBytesConfig`) | 26 | 0,4 |
| `pip install` | 15 | 0,3 |

Тобто ≈ 85 % часу - це тренування й оцінка моделей; власне тренування займає 1,7-4,9 хв на модель (рядок "Тривалість" у підсумковій таблиці),
решта - генерація відповідей на тестовій вибірці до й після донавчання та завантаження ваг з Hugging Face.

**Проблеми:**
- **Перший запуск обірвався** на тренуванні QLoRA (крок 59 зі 100): сесія Colab перервалась, і останні клітинки не виконались.
  Довелося перезапустити ноутбук повністю (Restart → Run all) - другий запуск дійшов до кінця.
- **Colab не зберіг виводи частини клітинок** у файлі (`run_lora("Qwen/Qwen3-4B")`, повне донавчання, перевірка адаптера, QLoRA):
  код у них виконався (вони є в таблиці часу, а їхні результати - у підсумковій таблиці розділу 9), але текст виводу у збереженому
  `.ipynb` порожній. Значення пам'яті 4B після завантаження (8,1 GB у fp16 і 2,7 GB у 4 бітах) у відповіді 23 взяті з першого запуску того самого коду.
- Дрібниці: попередження PEFT про перейменування `base_model_name_or_path` (той самий `peft_config` повторно використовується для різних моделей - на результат не впливає)
  і довге перше завантаження ваг 4B (≈ 8 GB) з Hub.

**23. Прокоментуйте вивід "Пам'ять GPU після завантаження". Чи очікуваним для Вас є результат?**

| модель | параметрів | пам'ять після завантаження | очікування: параметри × 2 байти (fp16) |
|---|---:|---:|---:|
| Qwen3-0.6B | 0.60 млрд | **1.2 GB** | 1.2 GB |
| Qwen3-1.7B | 1.72 млрд | **3.5 GB** | 3.4 GB |
| Qwen3-4B | 4.02 млрд | **8.1 GB** | 8.0 GB |
| Qwen3-4B у 4 бітах (QLoRA, NF4) | 4.02 млрд | **2.7 GB** | ≈ 2.6 GB (див. нижче) |

**Так, результат очікуваний.** Моделі завантажені у fp16, тобто 2 байти на параметр, і пам'ять майже точно дорівнює "кількість параметрів × 2".
Це лише ваги. Активації, KV-кеш під час генерації, градієнти й стани оптимізатора під час тренування додаються зверху, тому пікова пам'ять
під час LoRA більша (1.4 / 3.8 / 8.75 GB).

Для QLoRA: bitsandbytes квантизує лише лінійні шари, а таблиця ембеддингів (151 936 токенів × 2 560 = 0,39 млрд параметрів, у Qwen3-4B вона ж служить LM-головою)
лишається у fp16. Виходить ≈ 3,6 млрд × 0,5 байта ≈ 1,8 GB + 0,39 млрд × 2 байти ≈ 0,8 GB + масштаби блоків ≈ **2,7 GB**, точно як у виводі.
Тобто ваги 4B вдалося стиснути в ≈ 3 рази.
Видно й те, чому повне донавчання 4B неможливе на T4: вже самі ваги займають 8 з 15 GB, а потрібно ще ≈ 8 байтів на параметр для стану Adam.

**24. Що робить рядок `with torch.no_grad()` у функції `classify()`?**

(У цьому ноутбуці він стоїть у `classify_batch()`, яку викликає `classify()`.) Він **вимикає autograd** - механізм PyTorch, який
під час прямого проходу записує граф обчислень і зберігає проміжні активації, щоб потім порахувати градієнти.
Під час інференсу градієнти не потрібні, тож у блоці `no_grad`:
- не зберігаються активації для зворотного проходу → **значно менше пам'яті GPU** (важливо при генерації батчами по 32);
- немає накладних витрат на побудову графа → **швидше**;
- немає ризику випадково змінити ваги чи накопичити градієнти.

На результат (відповіді моделі) він не впливає.

**25. Для чого у `classify()` задано `do_sample=False`?**

`do_sample=False` - **жадібне декодування**: на кожному кроці береться найімовірніший токен, без випадковості. Для класифікації це потрібно, бо:
- є одна правильна відповідь, а не "творчий" текст: випадкова вибірка могла б обрати менш імовірну (неправильну) мітку
  або зламати JSON (зайва лапка, вигадана назва наміру);
- **відтворюваність**: та сама модель на тому самому зверненні завжди дає ту саму відповідь, тож метрики "до" і "після" чесно порівнюються
  між запусками, а не залежать від везіння.

**26. В демо 1 було натреновано класифікатор з базової моделі за допомогою заміни голови. В демо 3 в ході інференсу моделі також відбувається класифікація звернень. Чому не можна застосувати такий же підхід?**

Технічно для Qwen3 голову замінити можна (`AutoModelForSequenceClassification` підтримує і декодерні моделі - береться вектор останнього токена).
Але в демо 3 цього не роблять, бо:
- **вихід - не одна мітка, а структурована відповідь** `{"category", "intent"}` у JSON, яку читає програма. Новою головою
  довелось би робити дві голови (11 і 27 класів) і самим збирати JSON; генеративна модель видає готовий формат.
- **заміна голови "ламає" LLM**: LM-голова (прогноз наступного токена по всьому словнику) замінюється лінійним шаром на N класів -
  модель більше не вміє генерувати текст, відповідати на інші питання, пояснювати свою відповідь.
- **гнучкість**: щоб додати новий намір чи поле у відповіді, з SFT достатньо додати приклади; з головою треба міняти архітектуру й тренувати її заново.
- **LoRA-адаптер** можна вимкнути чи замінити іншим на тій самій базовій моделі; одна модель обслуговує кілька задач.
- з SFT можна оцінити ту саму модель **до донавчання з промптом** - для класифікатора з випадково ініціалізованою головою "до" не існує.

Тобто для простої класифікації голова - дешевий і хороший варіант (демо 1), а SFT потрібен, коли задача - **генерація у заданому форматі**.

**27. Порівняйте кількість токенів короткого й детального промпту (розділ 2). Як ця різниця вплине на вартість обробки 10 000 звернень на день?**

| | системний промпт | весь вхід моделі в середньому (шаблон + промпт + звернення) |
|---|---:|---:|
| короткий | 11 токенів | **38 токенів** |
| детальний (усі мітки) | 166 токенів | **193 токени** |

Детальний промпт робить **кожен запит у ≈ 5 разів довшим**: +155 токенів на звернення.

**10 000 звернень на день:** 10 000 × 155 = **1,55 млн зайвих токенів на день**, ≈ **46,5 млн на місяць**
(розрахунок у клітинці розділу 2). Обробка з детальним промптом коштує ≈ 193 / 38 ≈ **5 разів більше** за вхідні токени. Відповідь (JSON ≈ 15 токенів) однакова.
- **Через платний API:** вартість вхідних токенів зростає у 5 разів, бо API рахує всі токени, разом із системним промптом. Наприклад, за умовної ціни
  $0,50 за 1 млн вхідних токенів детальний промпт коштував би ≈ $0,97 на день замість ≈ $0,19 (≈ $29 проти ≈ $6 на місяць). Що дорожча модель, то більша різниця.
- **На своїй GPU** (як у лабі 4): у 5 разів більше роботи на етапі prefill і більший KV-кеш на кожен запит → менше запитів одночасно і довший час до першого токена.

Донавчання "переносить" перелік міток із промпту у ваги: модель після LoRA дає точність category 99-100 % з коротким промптом, а без донавчання
навіть з детальним промптом лише 38-88 %. Тож донавчена модель **і дешевша, і точніша**.

**28. Для чого в датасеті задано колонку `chat_template_kwargs` з `enable_thinking=False`? Що, на Вашу думку, сталося б із моделлю, якби її донавчали на цих прикладах у режимі `/think`?**

Колонка передає аргументи в `apply_chat_template` для кожного прикладу, коли TRL форматує дані. З `enable_thinking=False`
шаблон Qwen3 вставляє перед відповіддю асистента **порожній** блок `<think>\n\n</think>` - так само, як під час інференсу у
`classify_batch()` (там теж `enable_thinking=False`). Тобто тренувальні приклади виглядають **точно так, як запити, що модель отримає в роботі**,
і вона вчиться одразу після порожнього блоку міркувань видавати JSON.

Якби донавчати в режимі `/think` на прикладах, де відповідь - лише JSON без міркувань:
- шаблон не мав би порожнього блоку `<think></think>`, і модель вчилась би, що в режимі міркувань одразу після
  `<|im_start|>assistant` треба писати JSON, тобто **міркувати не треба**. Як попереджає демо 2, модель
  **розучилась би міркувати** і в інших задачах - режим `/think` став би фактично зламаним;
- з'явилась би **розбіжність форматів**: тренування в одному режимі, інференс в іншому (`enable_thinking=False`) → гірша точність;
- якби модель усе ж почала міркувати при інференсі, `max_new_tokens=60` обірвав би відповідь посеред `<think>`, і JSON не розібрався б.

**29. Що означає параметр `assistant_only_loss=True`? На яких токенах модель вчилася б без нього і чим це погано для нашої задачі?**

З `assistant_only_loss=True` TRL ставить мітку `-100` усім токенам, крім реплік асистента, тож функція втрат рахується
**лише на токенах відповіді** (JSON + `<|im_end|>`). Модель вчиться тільки одного: що відповідати на звернення.

Без нього loss рахувався б на **всіх** токенах: на службових токенах шаблону, системному промпті й тексті звернення клієнта. Чим це погано:
- звернення (≈ 20-30 токенів) + шаблон і системний промпт довші за відповідь (≈ 15 токенів) → **більшість градієнта** йшла б
  на те, щоб навчитися **писати звернення клієнтів** у стилі Bitext (з заповнювачами `{{Order Number}}`), а не класифікувати їх;
- системний промпт однаковий у всіх прикладах - модель просто запам'ятає його, це марна робота;
- на 100 кроках тренування сигнал для потрібної задачі слабший → гірша точність; validation loss стає менш інформативним
  (змішує якість класифікації з якістю "вгадування" тексту клієнта);
- модель може почати після відповіді генерувати нові "звернення", бо бачила їх як цільовий текст.

**30. Порівняйте LoRA і QLoRA за пам'яттю, часом тренування і якістю (розділ 9). У яких випадках Ви обрали б QLoRA?**

| | LoRA (Qwen3-4B, fp16) | QLoRA (Qwen3-4B, NF4) | різниця |
|---|---:|---:|---|
| пам'ять після завантаження | 8,1 GB | 2,7 GB | у 3 рази менше |
| **пікова пам'ять GPU під час тренування** | **8,75 GB** | **3,67 GB** | **у 2,4 раза менше** |
| **тривалість тренування** (100 кроків) | **3,67 хв** | **4,93 хв** | **на ≈ 34 % довше** |
| точність category | 100,0 % | 99,9 % | −0,1 п.п. |
| точність intent | 97,7 % | 97,8 % | +0,1 п.п. |
| чистий / валідний JSON | 100 % / 100 % | 100 % / 100 % | - |

**Пам'ять.** QLoRA тримає заморожені ваги в 4 бітах, тому пік тренування 4B-моделі (3,67 GB) навіть менший, ніж у LoRA на Qwen3-1.7B (3,83 GB).
**Час.** QLoRA повільніша на третину: на кожному прямому й зворотному проході bitsandbytes деквантизує блоки ваг з NF4 у fp16 перед множенням - це додаткова робота.
**Якість** практично однакова: різниця ±0,1 п.п. - це одне звернення з тестової вибірки, тобто в межах шуму. Втрата точності від 4-бітових ваг на цій задачі непомітна,
бо адаптер тренується у fp16 і "підлаштовується" під квантизовану модель.

**Коли я обрав би QLoRA:**
- модель **не влазить у GPU** у fp16: напр., 7-8B на T4 (16 GB ваг у fp16 > 15 GB) або 30B+ на одній 24-80 GB карті - тоді QLoRA єдиний варіант;
- хочеться **більший batch або довший контекст** на тій самій GPU - звільнена пам'ять іде на активації;
- безкоштовні / дешеві GPU (Colab T4), де пам'ять - головне обмеження, а зайві 30 % часу не критичні.

Якщо ж модель вільно поміщається у fp16 (як 4B на T4 тут), я обрав би **звичайну LoRA**: вона на третину швидша при тій самій якості
і не потребує квантизації при подальшому злитті адаптера з базовою моделлю.